In [1]:
import xml.etree.ElementTree as ET
import mujoco

In [31]:
urdf = ET.parse('mh5_revH.urdf')

In [32]:
ET.dump(urdf)

<robot name="mh5_robot">
  <link name="base_link" />
  <link name="chest">
    <visual>
      <origin rpy="0 0 0" xyz="0 0 0" />
      <geometry>
        <mesh filename="package://mh5_description/meshes/chest_revH.dae" />
      </geometry>
    </visual>
    <collision>
      <origin rpy="0 0 0" xyz="0 0 0" />
      <geometry>
        <mesh filename="package://mh5_description/meshes/chest_revH.dae" />
      </geometry>
    </collision>
    <inertial>
      <mass value="0.400" />
      <inertia ixx="0.000954830" ixy="-0.000001668" ixz="-0.000021416" iyy="0.000597536" iyz="0.000000148" izz="0.000637274" />
    </inertial>
  </link>
  <joint name="chest_fixed" type="floating">
    <parent link="base_link" />
    <child link="chest" />
    <origin rpy="0 0 0" xyz="0 0 0.203" />
  </joint>
  <joint name="head_y" type="revolute">
    <axis xyz="0 0 1" />
    <parent link="chest" />
    <child link="head_servo" />
    <origin rpy="0 0 0" xyz="-0.007076 0 0.19799999999999998" />
    <limit effo

In [33]:
# add mujoco tag
root = urdf.getroot()
assert root.tag == "robot"
root.insert(0, ET.fromstring("""
<mujoco>
    <compiler meshdir="meshes/" balanceinertia="true" discardvisual="false"/>
</mujoco>"""))

# replace DAE extensions with STL
for mesh in root.iter(tag='mesh'):
    filename = mesh.get('filename')
    if filename[-3:] == 'dae':
        mesh.set('filename', filename[:-3] + 'stl')

ET.dump(root)

<robot name="mh5_robot">
  <mujoco>
    <compiler meshdir="meshes/" balanceinertia="true" discardvisual="false" />
</mujoco><link name="base_link" />
  <link name="chest">
    <visual>
      <origin rpy="0 0 0" xyz="0 0 0" />
      <geometry>
        <mesh filename="package://mh5_description/meshes/chest_revH.stl" />
      </geometry>
    </visual>
    <collision>
      <origin rpy="0 0 0" xyz="0 0 0" />
      <geometry>
        <mesh filename="package://mh5_description/meshes/chest_revH.stl" />
      </geometry>
    </collision>
    <inertial>
      <mass value="0.400" />
      <inertia ixx="0.000954830" ixy="-0.000001668" ixz="-0.000021416" iyy="0.000597536" iyz="0.000000148" izz="0.000637274" />
    </inertial>
  </link>
  <joint name="chest_fixed" type="floating">
    <parent link="base_link" />
    <child link="chest" />
    <origin rpy="0 0 0" xyz="0 0 0.203" />
  </joint>
  <joint name="head_y" type="revolute">
    <axis xyz="0 0 1" />
    <parent link="chest" />
    <child link

In [34]:
mjmodel = mujoco.MjModel.from_xml_string(ET.tostring(root))

In [35]:
mujoco.mj_saveLastXML('mh5_revH.xml', mjmodel)

In [36]:
mujocoxml = ET.parse('mh5_revH.xml')
ET.dump(mujocoxml)

<mujoco model="mh5_robot">
  <compiler angle="radian" meshdir="meshes/" />

  <asset>
    <mesh name="chest_revH" file="chest_revH.stl" />
    <mesh name="2XL430_1_idle_revC" file="2XL430_1_idle_revC.stl" />
    <mesh name="head_revH" file="head_revH.stl" />
    <mesh name="frame04_revE" file="frame04_revE.stl" />
    <mesh name="upper_arm" file="upper_arm.stl" />
    <mesh name="frame02_revC" file="frame02_revC.stl" />
    <mesh name="left_lower_arm" file="left_lower_arm.stl" />
    <mesh name="gripper" file="gripper.stl" />
    <mesh name="right_lower_arm" file="right_lower_arm.stl" />
    <mesh name="2XL430_2_idles_revC" file="2XL430_2_idles_revC.stl" />
    <mesh name="thigh_revE" file="thigh_revE.stl" />
    <mesh name="foot_revH" file="foot_revH.stl" />
  </asset>

  <worldbody>
    <body name="chest" pos="0 0 0.203">
      <inertial pos="0 0 0" quat="0.707094 0.706321 0.022059 -0.0253348" mass="0.4" diaginertia="0.000956276 0.000635836 0.000597528" />
      <joint name="chest_fi

In [37]:
# for the time being the joint names are hardcoded
# better read them from the controller config YAML in the future
actuators = [
    "head_p", "head_y",
    "l_sho_p", "l_sho_r", "l_elb_r", "l_elb_p", "l_gripper",
    "r_sho_p", "r_sho_r", "r_elb_r", "r_elb_p", "r_gripper",
    "l_hip_r", "l_hip_p", "l_kne_p", "l_kne_y", "l_ank_p", "l_ank_r",
    "r_hip_r", "r_hip_p", "r_kne_p", "r_kne_y", "r_ank_p", "r_ank_r"
    ]

# actuators_xml = f"""<actuator>
#   {['<position name="'+joint+'" joint="' + joint + '"/>\n'  for joint in actuators]}
# </actuator>
# """
def make_one_position_xml(name: str) -> str:
    return '    <position name="' + name + '" joint="' + name + '"/>\n'

def make_all_positions_xml(names: list[str]) -> str:
    xml = ""
    for name in names:
        xml += make_one_position_xml(name)
    return xml

actuators_xml = "  <actuator>\n" + make_all_positions_xml(actuators) + "  </actuator>\n"

# add actuators
mujocoroot = mujocoxml.getroot()
mujocoroot.append(ET.fromstring(actuators_xml))

# add defaults
mujocoroot.append(ET.fromstring("""
  <default>
    <joint damping="1.084" armature="0.045" frictionloss="0.03"/>
    <position kp="21.1"/>
  </default>
"""))


In [38]:
ET.dump(mujocoxml)

<mujoco model="mh5_robot">
  <compiler angle="radian" meshdir="meshes/" />

  <asset>
    <mesh name="chest_revH" file="chest_revH.stl" />
    <mesh name="2XL430_1_idle_revC" file="2XL430_1_idle_revC.stl" />
    <mesh name="head_revH" file="head_revH.stl" />
    <mesh name="frame04_revE" file="frame04_revE.stl" />
    <mesh name="upper_arm" file="upper_arm.stl" />
    <mesh name="frame02_revC" file="frame02_revC.stl" />
    <mesh name="left_lower_arm" file="left_lower_arm.stl" />
    <mesh name="gripper" file="gripper.stl" />
    <mesh name="right_lower_arm" file="right_lower_arm.stl" />
    <mesh name="2XL430_2_idles_revC" file="2XL430_2_idles_revC.stl" />
    <mesh name="thigh_revE" file="thigh_revE.stl" />
    <mesh name="foot_revH" file="foot_revH.stl" />
  </asset>

  <worldbody>
    <body name="chest" pos="0 0 0.203">
      <inertial pos="0 0 0" quat="0.707094 0.706321 0.022059 -0.0253348" mass="0.4" diaginertia="0.000956276 0.000635836 0.000597528" />
      <joint name="chest_fi

In [39]:
mujocoxml.write("mh5_revH.xml")